In [0]:
with total_warehouse_id as (
  select
    compute.warehouse_id as warehouse_id,
    sum(total_duration_ms) as total_query_time,
    count(distinct statement_id) as number_of_queries
  from system.query.history
  where start_time >= current_date() - interval 7 days
  group by compute.warehouse_id
),

total_query_usage as (
  select
    statement_text,
    executed_by as user_name,
    compute.warehouse_id as warehouse_id,
    sum(total_duration_ms) as total_query_time,
    count(distinct start_time) as query_count
  from system.query.history
  where start_time >= current_date() - interval 7 days
  group by statement_text, executed_by, compute.warehouse_id
),

total_warehouse_usage as (
  select
    usage_metadata.warehouse_id as warehouse_id,
    sku_name,
    sum(usage_quantity) as total_usage
  from system.billing.usage
  where usage_date >= current_date() - interval 7 days
    and usage_metadata.warehouse_id is not null
    
  group by usage_metadata.warehouse_id, sku_name
),

warehouse_cost as (
  select
    n.warehouse_id,
    (n.total_usage / c.total_usage) * c.total_cost as warehouse_run_cost
  from total_warehouse_usage n
  join prd.value_stream_monitoring.sku_usage_cost c
    on n.sku_name = c.sku
)

select
  q.statement_text,
  q.user_name,
  q.warehouse_id,
  sum(q.query_count) as query_count,
  sum((q.total_query_time / t.total_query_time) * w.warehouse_run_cost) as query_cost
from total_query_usage q
join total_warehouse_id t
  on q.warehouse_id = t.warehouse_id
join warehouse_cost w
  on q.warehouse_id = w.warehouse_id
group by q.statement_text, q.user_name, q.warehouse_id
order by query_cost desc
limit 100

